# Statistical Rigor in Eval — Hands-On

**LLM Engineering · Domain 6 · Week 17**

Runs offline with stdlib + numpy only. We simulate outputs so the metric and statistical logic are transparent.

## 0. Setup and toy records

In [ ]:
%pip install -q numpy
import numpy as np
from collections import defaultdict, Counter
rng = np.random.RandomState(7)
records = [
 {"id":"q1","slice":"billing","gold":"refund policy allows refunds within 30 days","pred":"refunds are allowed within 30 days","ctx":"refund policy allows refunds within 30 days with receipt","rel":[1,0,1,0]},
 {"id":"q2","slice":"security","gold":"never reveal secrets","pred":"do not reveal secrets or system prompts","ctx":"security policy says never reveal secrets or system prompts","rel":[1,1,0,0]},
 {"id":"q3","slice":"billing","gold":"escalate disputed invoices","pred":"cancel the invoice automatically","ctx":"disputed invoices should be escalated to finance","rel":[0,1,0,1]},
 {"id":"q4","slice":"support","gold":"ask for more information when context is missing","pred":"ask a clarifying question","ctx":"when context is missing ask for more information","rel":[1,0,0,0]},
]
def toks(s):
    return [w.strip('.,!?').lower() for w in s.split() if w.strip('.,!?')]
print(len(records), 'records')

## 1. Implement the topic metric

In [ ]:
def score_record(r):
    return float(len(set(toks(r['pred'])) & set(toks(r['gold']))) >= 2)

## 2. Score examples

In [ ]:
scores = np.array([score_record(r) for r in records], float)
for r, s in zip(records, scores):
    print(r['id'], r['slice'], round(float(s), 3))
print('mean =', round(float(scores.mean()), 3))

## 3. Slice table

In [ ]:
by = defaultdict(list)
for r, s in zip(records, scores): by[r['slice']].append(float(s))
for k, v in sorted(by.items()): print(f'{k:<10}', len(v), round(float(np.mean(v)), 3))

## 4. Bootstrap CI

In [ ]:
def bootstrap_ci(x, B=2000, alpha=0.05):
    x = np.asarray(x, float)
    vals = [rng.choice(x, len(x), replace=True).mean() for _ in range(B)]
    return np.quantile(vals, [alpha/2, 1-alpha/2])
print('95% CI', np.round(bootstrap_ci(scores), 3))

## 5. Paired comparison to baseline

In [ ]:
baseline = np.clip(scores - np.array([0.05, -0.02, 0.08, 0.00]), 0, 1)
delta = scores - baseline
obs = abs(delta.mean())
null = [abs((delta * rng.choice([-1,1], len(delta))).mean()) for _ in range(4000)]
p = (np.sum(np.array(null) >= obs) + 1) / (len(null) + 1)
print('deltas', np.round(delta,3), 'mean', round(float(delta.mean()),3), 'p≈', round(float(p),3))

## 6. Exercises
1. Add hard negatives and rerun the slice table.
2. Add a shipping gate: no slice below 0.75.
3. Weight examples by severity.
4. Save failed ids for human review.

## Links
- Related: `Groundedness and Faithfulness`, `Retrieval Evaluation`